# SageMaker Processing — Removing Duplicates

This notebook demonstrates how to use a SageMaker Processing job to detect and remove duplicate rows from a dataset.

The dataset is deliberately created with duplicate rows. The processing script identifies them, removes them, and writes a before/after report alongside the cleaned data.

## Steps
1. [Setup](#1-setup)
2. [Create sample data with duplicates and upload to S3](#2-create-sample-data-with-duplicates-and-upload-to-s3)
3. [Write the deduplication script](#3-write-the-deduplication-script)
4. [Upload the script to S3](#4-upload-the-script-to-s3)
5. [Run the Processing job](#5-run-the-processing-job)
6. [Check the output](#6-check-the-output)

## 1. Setup

In [ ]:
import boto3

# Clients
sm_client  = boto3.client("sagemaker")
s3_client  = boto3.client("s3")
sts_client = boto3.client("sts")

# Account and region
identity   = sts_client.get_caller_identity()
account_id = identity["Account"]
region     = boto3.Session().region_name

# Role ARN — reconstructed from STS assumed-role ARN, no iam:GetRole needed
assumed_role_arn = identity["Arn"]
role_name = assumed_role_arn.split("/")[1]
role = f"arn:aws:iam::{account_id}:role/{role_name}"

# S3 bucket
bucket = f"sagemaker-{region}-{account_id}"
prefix = "processing-dedup-demo"

try:
    if region == "us-east-1":
        s3_client.create_bucket(Bucket=bucket)
    else:
        s3_client.create_bucket(
            Bucket=bucket,
            CreateBucketConfiguration={"LocationConstraint": region}
        )
    print(f"Created bucket: s3://{bucket}")
except s3_client.exceptions.BucketAlreadyOwnedByYou:
    print(f"Bucket already exists: s3://{bucket}")

print(f"\nRegion  : {region}")
print(f"Account : {account_id}")
print(f"Role    : {role}")
print(f"Bucket  : s3://{bucket}/{prefix}")

## 2. Create sample data with duplicates and upload to S3

We create a small employee dataset and deliberately add duplicate rows so we can prove the processing job removed them.

In [ ]:
import pandas as pd

# Original clean records
original_records = [
    {"employee_id": 101, "name": "Alice",   "department": "Engineering", "salary": 75000},
    {"employee_id": 102, "name": "Bob",     "department": "Marketing",   "salary": 55000},
    {"employee_id": 103, "name": "Charlie", "department": "Engineering", "salary": 90000},
    {"employee_id": 104, "name": "Diana",   "department": "HR",          "salary": 60000},
    {"employee_id": 105, "name": "Eve",     "department": "Marketing",   "salary": 72000},
]

# Deliberately duplicated rows (exact copies of existing records)
duplicate_records = [
    {"employee_id": 102, "name": "Bob",     "department": "Marketing",   "salary": 55000},  # duplicate of row 2
    {"employee_id": 104, "name": "Diana",   "department": "HR",          "salary": 60000},  # duplicate of row 4
]

df_raw = pd.DataFrame(original_records + duplicate_records)
df_raw.to_csv("employees_with_duplicates.csv", index=False)

print(f"Dataset has {len(df_raw)} rows ({len(duplicate_records)} are duplicates)\n")
print(df_raw.to_string(index=False))
print("\nRows marked as duplicate:")
print(df_raw[df_raw.duplicated()].to_string(index=False))

In [ ]:
# Upload raw data to S3
input_key = f"{prefix}/input/employees_with_duplicates.csv"
s3_client.upload_file("employees_with_duplicates.csv", bucket, input_key)
input_s3_uri = f"s3://{bucket}/{input_key}"
print(f"Uploaded to: {input_s3_uri}")

## 3. Write the deduplication script

This script runs inside the SageMaker Processing container. It:
- Loads the raw CSV
- Identifies and logs every duplicate row
- Removes duplicates
- Writes the cleaned CSV and a plain-text report to `/opt/ml/processing/output`

In [ ]:
%%writefile dedup_script.py
import pandas as pd
import os

print("=" * 50)
print("SageMaker Processing — Deduplication Job")
print("=" * 50)

# Load input data
input_path = "/opt/ml/processing/input/employees_with_duplicates.csv"
df = pd.read_csv(input_path)

rows_before = len(df)
print(f"\nRows loaded  : {rows_before}")
print("\nFull dataset before deduplication:")
print(df.to_string(index=False))

# Identify duplicates before removing them
duplicate_mask = df.duplicated(keep="first")   # mark all but the first occurrence
duplicates_df  = df[duplicate_mask]
num_duplicates = len(duplicates_df)

print(f"\nDuplicate rows found: {num_duplicates}")
if num_duplicates > 0:
    print(duplicates_df.to_string(index=False))

# Remove duplicates — keep the first occurrence of each row
df_clean = df.drop_duplicates(keep="first").reset_index(drop=True)
rows_after = len(df_clean)

print(f"\nRows after deduplication: {rows_after}")
print("\nCleaned dataset:")
print(df_clean.to_string(index=False))

# Write outputs
os.makedirs("/opt/ml/processing/output", exist_ok=True)

# 1. Cleaned CSV
clean_path = "/opt/ml/processing/output/employees_clean.csv"
df_clean.to_csv(clean_path, index=False)
print(f"\nCleaned CSV written to  : {clean_path}")

# 2. Plain-text report
report_lines = [
    "=" * 50,
    "Deduplication Report",
    "=" * 50,
    f"Rows before : {rows_before}",
    f"Duplicates  : {num_duplicates}",
    f"Rows after  : {rows_after}",
    "",
    "Duplicate rows that were removed:",
]

if num_duplicates > 0:
    report_lines.append(duplicates_df.to_string(index=False))
else:
    report_lines.append("  None found.")

report_lines += [
    "",
    "Cleaned dataset:",
    df_clean.to_string(index=False),
]

report_path = "/opt/ml/processing/output/dedup_report.txt"
with open(report_path, "w") as f:
    f.write("\n".join(report_lines))
print(f"Report written to       : {report_path}")
print("\nJob complete.")

## 4. Upload the script to S3

In [ ]:
script_key = f"{prefix}/code/dedup_script.py"
s3_client.upload_file("dedup_script.py", bucket, script_key)
script_s3_uri = f"s3://{bucket}/{script_key}"
print(f"Script uploaded to: {script_s3_uri}")

## 5. Run the Processing job

In [ ]:
import time

sklearn_image  = f"683313688378.dkr.ecr.{region}.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3"
job_name       = f"dedup-demo-{int(time.time())}"
output_s3_uri  = f"s3://{bucket}/{prefix}/output"

print(f"Job name  : {job_name}")
print(f"Output    : {output_s3_uri}")
print("Submitting job...")

response = sm_client.create_processing_job(
    ProcessingJobName=job_name,
    RoleArn=role,
    AppSpecification={
        "ImageUri": sklearn_image,
        "ContainerEntrypoint": ["python3", "/opt/ml/processing/code/dedup_script.py"],
    },
    ProcessingInputs=[
        {
            "InputName": "input-data",
            "S3Input": {
                "S3Uri": input_s3_uri,
                "LocalPath": "/opt/ml/processing/input",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
            },
        },
        {
            "InputName": "code",
            "S3Input": {
                "S3Uri": script_s3_uri,
                "LocalPath": "/opt/ml/processing/code",
                "S3DataType": "S3Prefix",
                "S3InputMode": "File",
            },
        },
    ],
    ProcessingOutputConfig={
        "Outputs": [
            {
                "OutputName": "cleaned-data",
                "S3Output": {
                    "S3Uri": output_s3_uri,
                    "LocalPath": "/opt/ml/processing/output",
                    "S3UploadMode": "EndOfJob",
                },
            }
        ]
    },
    ProcessingResources={
        "ClusterConfig": {
            "InstanceCount": 1,
            "InstanceType": "ml.t3.medium",
            "VolumeSizeInGB": 10,
        }
    },
    StoppingCondition={"MaxRuntimeInSeconds": 300},
)

print(f"Job submitted. ARN: {response['ProcessingJobArn']}")

In [ ]:
import time

print("Waiting for job to complete (polling every 30s)...\n")
while True:
    status     = sm_client.describe_processing_job(ProcessingJobName=job_name)
    job_status = status["ProcessingJobStatus"]
    print(f"  Status: {job_status}")
    if job_status in ["Completed", "Failed", "Stopped"]:
        break
    time.sleep(30)

if job_status == "Completed":
    print("\nJob completed successfully.")
else:
    print(f"\nJob failed: {status.get('FailureReason', 'No reason provided')}")

## 6. Check the output

Download both output files from S3 and display them side by side.

In [ ]:
import pandas as pd

# Download cleaned CSV
s3_client.download_file(bucket, f"{prefix}/output/employees_clean.csv", "employees_clean.csv")
df_clean = pd.read_csv("employees_clean.csv")

# Download report
s3_client.download_file(bucket, f"{prefix}/output/dedup_report.txt", "dedup_report.txt")
with open("dedup_report.txt") as f:
    report = f.read()

# Show before and after
print(f"BEFORE — {len(df_raw)} rows (including duplicates):")
print(df_raw.to_string(index=False))

print(f"\nAFTER — {len(df_clean)} rows (duplicates removed):")
print(df_clean.to_string(index=False))

print(f"\nRows removed: {len(df_raw) - len(df_clean)}")

In [ ]:
# Print the full report written by the processing job
print(report)

## Summary

| | Value |
|---|---|
| Rows before processing | 7 |
| Duplicate rows detected | 2 |
| Rows after processing | 5 |
| Output files | `employees_clean.csv`, `dedup_report.txt` |

**The processing job ran entirely inside a managed SageMaker container** — no local compute was used for the deduplication itself. The cleaned data and report were written back to S3 automatically when the job completed.

**To adapt this for a GenAI pipeline:** the same pattern applies to deduplicating document chunks before embedding — preventing identical content from polluting your vector store and wasting embedding API calls.